In [2]:
import numpy as np
from ase.visualize import view
from ase import Atoms

# 1D H2 Chain

In [3]:
def make_h2_1D(intraH=1.0, interH=1.5, nx=1, vacuum=17.5):
    Lx = nx * (intraH + interH)
    cell = np.diag([Lx, vacuum, vacuum])
    y0 = vacuum / 2.0
    z0 = vacuum / 2.0
    positions = []
    symbols = []
    for i in range(nx):
        x0 = i * (intraH + interH)
        positions.append([x0,        y0, z0])   # H1
        positions.append([x0+intraH, y0, z0])   # H2
        symbols += ['H', 'H']
    atoms = Atoms(symbols=symbols, positions=positions, cell=cell, pbc=(True, False, False))
    atoms.center(axis=(1, 2))
    return atoms

In [4]:
h1D = make_h2_1D(intraH=0.74, interH=1.5, nx=5, vacuum=17.5)
view(h1D, viewer="x3d")
# h1D.write("POSCAR", format="vasp")

# 2D H2 Lattice

In [5]:
def make_h2_2D(intraH=1.0, interH=1.5, nx=1, ny=1, vacuum=17.5):
    ax = nx * (intraH + interH)
    by = ny * (intraH + interH)
    cz = vacuum
    cell = np.diag([ax, by, cz])
    
    positions = []
    symbols = []
    for ix in range(nx):
        for iy in range(ny):
            x0 = ix * (intraH + interH)
            y0 = iy * (intraH + interH)
            positions.append([x0, y0, vacuum / 2.0])
            positions.append([x0 + intraH, y0, vacuum / 2.0])
            symbols += ['H', 'H']

    atoms = Atoms(symbols, positions=positions, cell=cell, pbc=True)
    atoms.center()
    return atoms

In [6]:
h2D = make_h2_2D(intraH=0.74, interH=1.5, nx=8, ny=8, vacuum=17.5)
# h2D.write("POSCAR", format="vasp")
view(h2D, viewer="x3d")

# 3D H2-Lattice

In [7]:
def make_h2_3D(intraH=1.0, interH=1.5, nx=1, ny=1, nz=1, vacuum=17.5):
    pitch = intraH + interH
    ax = nx * pitch
    by = ny * pitch
    cz = nz * pitch
    cell = np.diag([ax, by, cz])

    positions = []
    symbols = []

    for ix in range(nx):
        for iy in range(ny):
            for iz in range(nz):
                x0 = ix * pitch
                y0 = iy * pitch
                z0 = iz * pitch
                positions.append([x0,          y0, z0])
                positions.append([x0 + intraH, y0, z0])
                symbols += ['H', 'H']

    atoms = Atoms(symbols, positions=positions, cell=cell, pbc=True)
    atoms.center()
    return atoms

In [9]:
h3D = make_h2_3D(intraH=0.74, interH=1.5, nx=2, ny=2, nz=2, vacuum=17.5)
view(h3D, viewer="x3d")
# h3D.write("POSCAR", format="vasp")

# Helper function

In [32]:
from pyscf.pbc import tools

get_phase = tools.k2gamma.get_phase
group_by_conj_pairs = tools.k2gamma.group_by_conj_pairs

def get_mo_coeff_k2R(kmf, mo_coeff_guess, ncore, ncas, kmesh=None):
    '''
    Get the k-point mo_coeff from the real-space mo_coeff_R
    '''
    cell = kmf.cell
    kpts = kmf.kpts
    sk = kmf.get_ovlp(kpts=kpts)
    mo_coeff_k = [mo[:, ncore:ncore+ncas] for mo in mo_coeff_guess]
    mo_energy_k = np.hstack([kmf.mo_energy[k][ncore:ncore+ncas] for k in range(len(kpts))])

    # Sanity Check
    actshape = mo_coeff_k[0].shape
    assert all([mo.shape == actshape for mo in mo_coeff_k])
    E_g = np.hstack(mo_energy_k)
    C_k = np.asarray(mo_coeff_k)
    Nk, Nao, Nmo = C_k.shape

    scell, phase = get_phase(cell, kpts, kmesh)

    NR = phase.shape[0]

    k_conj_groups = group_by_conj_pairs(cell, kpts, return_kpts_pairs=False)
    k_phase = np.eye(Nk, dtype=np.complex128)
    r2x2 = np.array([[1., 1j], [1., -1j]]) * .5**.5
    pairs = [[k, k_conj] for k, k_conj in k_conj_groups
             if k_conj is not None and k != k_conj]
    for idx in np.array(pairs):
        k_phase[idx[:,None],idx] = r2x2

    # complex supercell (real-space) MOs
    mo_coeff_R = np.einsum('Rk,kum,kh->Ruhm', phase, C_k, k_phase)
    mo_coeff_R = mo_coeff_R.reshape(Nao*NR, Nk*Nmo)

    # sort by energy
    # E_sort_idx = np.argsort(E_g, kind='stable')
    # E_g = E_g[E_sort_idx]
    # mo_coeff_R = mo_coeff_R[:, E_sort_idx]

    # mo_phase can be computed with the (possibly complex) mo_coeff_R
    s_k = cell.pbc_intor('int1e_ovlp', kpts=kpts)
    s_k_g = np.einsum('kuv,Rk->kuRv', s_k, phase.conj()).reshape(Nk,Nao,NR*Nao)
    mo_phase = lib.einsum('kum,kuv,vi->kmi', C_k.conj(), s_k_g, mo_coeff_R)
    
    return scell, phase, mo_coeff_R, mo_phase

# Calculations

In [ ]:
from pyscf.pbc import gto as pgto, scf
from pyscf import lib
from pyscf.lo import orth

atoms3D = make_h2_3D(intraH=0.74, interH=1.5, nx = 1, ny = 1, nz=1, vacuum=17.5)
cell = pgto.Cell()
cell.a = atoms3D.cell.array
pos = atoms3D.get_positions()
sym = atoms3D.get_chemical_symbols()
cell.atom = [(sym[i], tuple(pos[i])) for i in range(len(sym))]
cell.basis = 'STO-6G'
cell.unit = 'Angstrom'
cell.max_memory = 100000
cell.ke_cutoff = 100
cell.precision = 1e-10
cell.verbose = lib.logger.INFO
# cell.output = 'h2_3D_CASCI.log'
cell.build()

kmesh = [1, 1, 3]
kpts = cell.make_kpts(kmesh)
kmf = scf.KRHF(cell, kpts=kpts).density_fit('def2-SVP-JKFIT')
kmf.exxdiv = None
kmf.kernel()


System: uname_result(system='Linux', node='Saumil-lambda-quad', release='6.8.0-90-generic', version='#91~22.04.1-Ubuntu SMP PREEMPT_DYNAMIC Thu Nov 20 15:20:45 UTC 2', machine='x86_64')  Threads 36
Python 3.11.5 (main, Sep 11 2023, 13:54:46) [GCC 11.2.0]
numpy 1.26.4  scipy 1.15.3  h5py 3.13.0
Date: Mon Mar  2 11:02:09 2026
PySCF version 2.10.0
PySCF path  /user/bhavnesh/anaconda3/lib/python3.11/site-packages/pyscf

[CONFIG] conf_file /user/bhavnesh/.pyscf_conf.py
[INPUT] verbose = 4
[INPUT] num. atoms = 2
[INPUT] num. electrons = 2
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = Angstrom
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 H      0.750000000000   1.120000000000   1.120000000000 AA    1.417294593424   2.116493259513   2.116493259513 Bohr   0.0
[INPUT]  2 H      1.490000000000   1.120000000000   1.12000000000

lattice sum = 739 cells
precision = 1e-10
pseudo = None
ke_cutoff = 100
    = [21 21 21] mesh (9261 PWs)




******** <class 'pyscf.pbc.scf.khf.KRHF'> ********
method = KRHF
initial guess = minao
damping factor = 0
level_shift factor = 0
DIIS = <class 'pyscf.scf.diis.CDIIS'>
diis_start_cycle = 1
diis_space = 8
diis_damp = 0
SCF conv_tol = 1e-08
SCF conv_tol_grad = None
SCF max_cycles = 50
direct_scf = True
direct_scf_tol = 1e-13
chkfile to save SCF result = /tmp/nep/tmpnvomxg4w
max_memory 100000 MB (current use 297 MB)


******** PBC SCF flags ********
N kpts = 3
Exchange divergence treatment (exxdiv) = None
DF object = <pyscf.pbc.df.df.GDF object at 0x79cfb0536d10>
Set gradient conv threshold to 0.0001


******** <class 'pyscf.pbc.df.rsdf_builder._RSNucBuilder'> ********
mesh = [9 9 9] (729 PWs)
ke_cutoff = 17.62608876862628
omega = 0.5192005323403229
exclude_d_aux = False
exclude_dd_block = True
j2c_eig_always = False
has_long_range = True


******** <class 'pyscf.pbc.df.df.GDF'> ********
auxbasis = def2-SVP-JKFIT
exp_to_discard = None
_cderi_to_save = /tmp/nep/tmpkemagqhr
len(kpts) = 3
D

-1.2167116536505955

In [45]:
scell, phase, mo_coeff_R, mo_phase = get_mo_coeff_k2R(kmf, mo_coeff_guess=kmf.mo_coeff, ncore=0, ncas=2)

In [46]:
ovlp = scell.pbc_intor('int1e_ovlp')
ao2lo = orth.orth_ao(scell, method='meta_lowdin')
ao2lo.shape

(6, 6)

In [47]:
from functools import reduce
lo_coeff = reduce(np.dot, (ao2lo.conj().T, ovlp, mo_coeff_R))

In [48]:
lo_coeff.dtype

dtype('complex128')

In [49]:
from pyscf.tools import molden
molden.from_mo(scell, 'h2_3D_CASCI.molden', lo_coeff)

/user/bhavnesh/anaconda3/lib/python3.11/site-packages/pyscf/tools/molden.py:77: ComplexWarning: Casting complex values to real discards the imaginary part
  fout.write(' %3d    %18.14g\n' % (i+1, mo_coeff[j,imo]))


In [50]:
scell.atom

[('H', (1.4172945934237966, 2.1164932595128696, 2.1164932595128696)),
 ('H', (2.8156919256019424, 2.1164932595128696, 2.1164932595128696)),
 ('H', (1.4172945934237966, 2.1164932595128696, 6.349479778538608)),
 ('H', (2.8156919256019424, 2.1164932595128696, 6.349479778538608)),
 ('H', (1.4172945934237966, 2.1164932595128696, 10.582466297564348)),
 ('H', (2.8156919256019424, 2.1164932595128696, 10.582466297564348))]